# gepa

gepa 0.1.4 is the GEPA optimizer library: it evolves the text parts of a system (prompts, instructions) by running
it, asking a language model to reflect on what went wrong, and keeping the candidates that score best on a Pareto
front. DSPy's `dspy.GEPA` optimizer calls it. These cells run a complete optimization with a toy adapter and a
deterministic stand-in for the reflecting model (no network), and check the pieces a user plugs together.

In [ ]:
# Example 1: the public surface a user starts from.
import gepa
from gepa.core.adapter import EvaluationBatch, GEPAAdapter

print(gepa.optimize.__module__, EvaluationBatch.__name__, GEPAAdapter.__name__)
assert callable(gepa.optimize)
batch = EvaluationBatch(outputs=["a"], scores=[1.0], trajectories=None)
assert batch.scores == [1.0] and batch.outputs == ["a"]

In [ ]:
# Example 2: a full optimization run -- a toy adapter, a fixed "reflection" reply, and a metric-call budget.
import gepa
from gepa.core.adapter import EvaluationBatch


class Shout:
    propose_new_texts = None
    def evaluate(self, batch, candidate, capture_traces=False):
        outs = [x.upper() if "SHOUT" in candidate["instructions"] else x for x in batch]
        traces = [{"in": x, "out": o} for x, o in zip(batch, outs)] if capture_traces else None
        return EvaluationBatch(outputs=outs, scores=[1.0 if o.isupper() else 0.0 for o in outs], trajectories=traces)
    def make_reflective_dataset(self, candidate, eval_batch, components):
        return {c: [{"Inputs": t["in"], "Generated Outputs": t["out"], "Feedback": "use capitals"} for t in eval_batch.trajectories] for c in components}


calls = []
result = gepa.optimize(seed_candidate={"instructions": "Answer politely."}, trainset=["hello", "world"], valset=["abc", "xyz"],
                       adapter=Shout(), reflection_lm=lambda prompt: calls.append(prompt) or "```\nSHOUT every answer.\n```",
                       max_metric_calls=20, reflection_minibatch_size=2)
print(result.best_candidate, result.val_aggregate_scores, len(calls))
assert "SHOUT" in result.best_candidate["instructions"] and max(result.val_aggregate_scores) == 1.0 and len(calls) >= 1

In [ ]:
# Example 3: the seed candidate alone scores 0, so the gain in Example 2 came from the proposal.
from gepa.core.adapter import EvaluationBatch


class Shout:
    def evaluate(self, batch, candidate, capture_traces=False):
        outs = [x.upper() if "SHOUT" in candidate["instructions"] else x for x in batch]
        return EvaluationBatch(outputs=outs, scores=[1.0 if o.isupper() else 0.0 for o in outs], trajectories=None)


seed = Shout().evaluate(["abc", "xyz"], {"instructions": "Answer politely."})
best = Shout().evaluate(["abc", "xyz"], {"instructions": "SHOUT every answer."})
print(seed.scores, best.scores)
assert seed.scores == [0.0, 0.0] and best.scores == [1.0, 1.0]

In [ ]:
# Example 4: the candidate selectors and data loaders gepa builds a run from.
from gepa.core.data_loader import ListDataLoader
from gepa.strategies.candidate_selector import CurrentBestCandidateSelector, ParetoCandidateSelector

loader = ListDataLoader(["a", "b", "c"])
ids = list(loader.all_ids())
print(ids, loader.fetch(ids[:2]), len(loader))
assert len(loader) == 3 and loader.fetch(ids[:2]) == ["a", "b"]
print(ParetoCandidateSelector.__name__, CurrentBestCandidateSelector.__name__)

In [ ]:
# Example 5: DSPy's GEPA optimizer is built on this package.
import dspy
import gepa

print(dspy.GEPA.__module__, gepa.__name__)
assert dspy.GEPA.__module__ == "dspy.teleprompt.gepa.gepa"
assert "gepa" in dspy.teleprompt.gepa.gepa.__dict__ or hasattr(dspy.teleprompt.gepa.gepa, "GEPA")